# Chapter 8: Numerical Methods

Companion notebook for *The Math That Powers AI* (2nd ed.).

Floating-point arithmetic is not real arithmetic. This notebook walks through the chapter's numerical-stability toolkit using the `mathpowersai.numerics` package: the naive vs. stable softmax, the log-sum-exp trick, the FP32 summation stall, machine epsilon, Kahan (compensated) summation, condition numbers, and Newton-Cotes quadrature.

All functions are imported from the package; nothing is redefined here.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

## 1. Naive vs. stable softmax

The softmax is defined as

$$\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}.$$

Implemented literally, this overflows for large logits: for $z = [1000, 1001, 999]$, $e^{1000} = \infty$ in floating point, and $\infty/\infty = \mathrm{NaN}$.

The fix is to subtract the maximum first, which is mathematically a no-op because
$e^{z_i - c} / \sum_j e^{z_j - c} = e^{z_i} / \sum_j e^{z_j}$, but keeps every exponent $\le 0$. The stable version returns $[0.245, 0.665, 0.090]$.

In [ ]:
from mathpowersai.numerics import naive_softmax, stable_softmax

z = np.array([1000.0, 1001.0, 999.0])

print("naive_softmax(z)  =", naive_softmax(z))
stable = stable_softmax(z)
print("stable_softmax(z) = [" + ", ".join(f"{p:.3f}" for p in stable) + "]")
print("sums to:", stable.sum())

## 2. The log-sum-exp trick

Log-probabilities require $\log \sum_i e^{x_i}$, which overflows directly once any $x_i \gtrsim 700$ in FP64. The same max-shift saves us:

$$\log \sum_i e^{x_i} = x_{\max} + \log \sum_i e^{x_i - x_{\max}}.$$

The shifted exponentials are all $\le 1$, so the result stays finite even for huge inputs. On small inputs it agrees with the direct computation.

In [ ]:
from mathpowersai.numerics import log_sum_exp

# Direct computation overflows ...
with np.errstate(over="ignore"):
    direct = np.log(np.sum(np.exp(np.array([1000.0, 1001.0]))))
print("direct  log(sum(exp([1000, 1001]))) =", direct)

# ... the trick stays finite.
print(f"log_sum_exp([1000, 1001])           = {log_sum_exp([1000.0, 1001.0]):.4f}")

# Sanity check on small inputs where both methods work:
small = np.array([0.5, 1.0, -0.5])
print(f"log_sum_exp([0.5, 1.0, -0.5]) = {log_sum_exp(small):.6f}"
      f" (direct: {np.log(np.sum(np.exp(small))):.6f})")

## 3. The FP32 summation stall at $2^{24}$

FP32 has a 23-bit mantissa, so consecutive integers are representable only up to $2^{24} = 16{,}777{,}216$. Once a naive running sum reaches that value, adding $1.0$ rounds back to the same number:

$$\mathrm{float32}(2^{24}) + \mathrm{float32}(1.0) = 2^{24}.$$

Summing $10^8$ copies of $1.0$ in FP32 therefore *stalls* at 16,777,216 instead of reaching 100,000,000 — a real failure mode for long FP32 accumulations in training loops.

In [ ]:
from mathpowersai.numerics import fp32_stall_demo

demo = fp32_stall_demo()
print(f"np.float32(2**24)       = {demo['stall_value']:,.0f}")
print(f"np.float32(2**24) + 1.0 = {demo['after_adding_one']:,.0f}")
print("stalled:", demo["stalled"])
print("Summing 1e8 ones in FP32 naively stalls at 16,777,216.")

## 4. Machine epsilon

Machine epsilon is the gap between $1$ and the next representable float:

$$\varepsilon_{\mathrm{mach}} = \mathrm{fl}(1^+) - 1.$$

For FP64, $\varepsilon_{\mathrm{mach}} = 2^{-52} \approx 2.2 \times 10^{-16}$ (about 16 decimal digits); for FP32, $\varepsilon_{\mathrm{mach}} = 2^{-23} \approx 1.2 \times 10^{-7}$ (about 7 digits). Every rounded arithmetic operation can introduce a relative error up to $\varepsilon_{\mathrm{mach}}/2$.

In [ ]:
from mathpowersai.numerics import machine_epsilon

eps64 = machine_epsilon(np.float64)
eps32 = machine_epsilon(np.float32)
print(f"FP64: {eps64:.6e}  (2**-52 = {2.0**-52:.6e})")
print(f"FP32: {eps32:.6e}  (2**-23 = {2.0**-23:.6e})")
assert eps64 == 2.0 ** -52
assert eps32 == 2.0 ** -23

## 5. Kahan (compensated) summation

When terms of very different magnitudes are added, the small term's low-order bits are lost: naively, $10^{16} + 1 - 10^{16} = 0$ in FP64. Kahan summation carries a correction term $c$ that recovers those lost bits (the package uses the Kahan-Babuska/Neumaier variant), so the same sum returns the exact answer $1$.

In [ ]:
from mathpowersai.numerics import kahan_sum

vals = np.array([1e16, 1.0, -1e16])

naive = sum(vals.tolist())
compensated = kahan_sum(vals)
print(f"naive sum(1e16 + 1 - 1e16) = {naive:.1f}")
print(f"kahan_sum(1e16 + 1 - 1e16) = {compensated:.1f}")
assert compensated == 1.0

## 6. Condition numbers

The 2-norm condition number of a matrix is the ratio of its extreme singular values:

$$\kappa_2(A) = \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}.$$

A problem with $\kappa = 10^k$ can lose up to $k$ digits of accuracy *no matter what algorithm you use*. $\kappa \approx 1$ is well-conditioned; $\kappa \approx 10^{16}$ is essentially singular in FP64. Below, the identity matrix is perfectly conditioned, while two nearly parallel rows blow $\kappa$ up to about $4 \times 10^4$ — meaning roughly 4 digits lost when solving $Ax = b$.

In [ ]:
from mathpowersai.numerics import condition_number

well = np.eye(2)
ill = np.array([[1.0, 1.0], [1.0, 1.0001]])

print(f"kappa(I)                     = {condition_number(well):.3e}  (well-conditioned)")
print(f"kappa([[1, 1], [1, 1.0001]]) = {condition_number(ill):.3e}  (ill-conditioned)")
print(f"-> up to {np.log10(condition_number(ill)):.1f} digits of accuracy can be lost")

## 7. Quadrature: trapezoid vs. Simpson

Newton-Cotes rules approximate $\int_a^b f(x)\,dx$ with $h = b - a$ and $m = (a+b)/2$:

- **Trapezoidal rule** (linear interpolation): $\frac{h}{2}\left[f(a) + f(b)\right]$, composite error $O(h^2)$.
- **Simpson's rule** (quadratic interpolation): $\frac{h}{6}\left[f(a) + 4f(m) + f(b)\right]$, composite error $O(h^4)$, exact for polynomials up to degree 3.

For $\int_0^2 x^2\,dx = 8/3$, the trapezoid rule overshoots (it returns 4), while Simpson's rule is exact because $x^2$ is a quadratic.

In [ ]:
from mathpowersai.numerics import simpson_rule, trapezoid_rule

f = lambda x: x ** 2
exact = 8.0 / 3.0

trap = trapezoid_rule(f, 0.0, 2.0)
simp = simpson_rule(f, 0.0, 2.0)
print(f"trapezoid_rule = {trap:.6f}  (error: {abs(trap - exact):.6f})")
print(f"simpson_rule   = {simp:.6f}  (error: {abs(simp - exact):.6f})")
print(f"exact 8/3      = {exact:.6f}")
assert abs(simp - exact) < 1e-12